In [0]:
# Delta Tables

In [0]:
# ACID Transactions -> Atomicity, Consistency, Isolation, Durability
# Time Travel -> Version Number, Timestamp value
# schema evolution

In [0]:
# delete from table
# where 

In [0]:
from pyspark.sql import functions as F

In [0]:
cus_df = spark.table("samples.bakehouse.sales_customers")
cus_df.write.format("delta").mode("overwrite").save("abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers")

In [0]:
cus_df.display()

In [0]:
cus_df = cus_df.filter(F.col("continent") == "Asia")
cus_df.display()

In [0]:
cus_df.write.format("delta").mode("overwrite").save("abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers")

In [0]:
cus_df_delta = (
    spark.read.format("delta")
    .load("abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers")
)

cus_df_delta.display()

In [0]:
%sql
DESCRIBE HISTORY delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers`

In [0]:
%sql
select * from delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers`;

In [0]:
%sql
select * from delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers` version as of 0;

In [0]:
%sql
select * from delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers` timestamp as of '2026-10-19T14:52:54.000+00:00';

In [0]:
%sql
select * from delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers`

In [0]:
%sql
restore delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers` version as of 0;

In [0]:
cus_df_delta = (
    spark.read.format("delta")
    .load("abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers")
)

cus_df_delta.display()

In [0]:
cus_df_delta = (
    cus_df_delta
    .withColumn(
        "test_gender", 
        F.when(
            F.col("gender") == "female", 0
        ).otherwise(1)
    )
)

cus_df_delta.display()

In [0]:
(
    cus_df_delta.write.format("delta")
    .mode("append")
    .save("abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers")
)

In [0]:
(
    cus_df_delta.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save("abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers")
)

In [0]:
cus_df_delta = (
    spark.read.format("delta")
    .load("abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers")
)

cus_df_delta.display()

In [0]:
%sql
DESCRIBE HISTORY delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers`

In [0]:
%sql
select * from delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers` version as of 2

### VACUUM 

In [0]:
%sql
VACUUM delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers` RETAIN 0 HOURS

In [0]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

In [0]:
%sql
VACUUM delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers` RETAIN 0 HOURS

### Optimize & z-order

In [0]:
cus_df = spark.table("samples.bakehouse.sales_customers")
cus_df = cus_df.repartition(50)
cus_df.write.format("delta").mode("append").save("abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers")

In [0]:
%sql
optimize delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers`;

In [0]:
%sql
desc history delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers`

In [0]:
%sql
optimize delta.`abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/delta_tables/customers`
zorder by (customerID);